# Dispersal Model Translation Part II (Semi-completed)

Lets create a basic dispersal model in Python using the Mesa framework.

- We start with step 1, which is to create a cell agent class with a cell location, that shouts out its location.
- Then we go create a visualization (see full func model)
- Then we go to step 2, which is to create a model class that initializes a 25x25 grid model, and let the agents shout out their location.
- Then you collect some model data (see full func model)


## Curent model description
This is a spatial dispersal model where agents represent organisms or particles that reproduce and spread across a landscape. Agents start clustered in a small corner area (5x5 cells) and each simulation step, they probabilistically "hatch" new offspring into neighboring empty cells based on their growth rate. Over time, the population expands outward from the initial cluster, creating a spreading pattern across the grid. The model allows to set number of agents, size of the grid, and growth rate.

## Import packages

In [1]:
import numpy as np
import pandas as pd
import mesa
from mesa.discrete_space import CellAgent, OrthogonalMooreGrid
from mesa.visualization import SolaraViz, make_plot_component, make_space_component

## Create Dispersal Agent Class

### Step 1: Create the DispersalAgent Class
Inherit from CellAgent and implement:
Constructor (__init__):
* Parameters: model, cell, growth_rate
* Store cell and growth_rate as instance variables
* Call super().__init__(model)
* For now just print `f"Hi I am cell agent # {self.unique_id} at position {self.cell.coordinate}"` as the disperse/hatch behavior

### Step2: Hatch method (hatch):
* Use growth_rate as probability: if self.random.random() < self.growth_rate:
Get neighbors: `list(self.cell.neighborhood.cells)`
* Find empty cells: `[cell for cell in neighbors if len(cell.agents) == 0]`
* If empty cells exist, pick one randomly and create new agent there

In [ ]:
def compute_dispersion(model):
    """Compute the dispersion of the agents."""
    positions = [agent.cell.coordinate for agent in model.agents]
    if positions:
        x_coords = [pos[0] for pos in positions]
        y_coords = [pos[1] for pos in positions]
        dispersion = np.std(x_coords) + np.std(y_coords)
    else:
        dispersion = 0
    return dispersion

# Set up an agent similar to the full func model
class DispersalAgent(CellAgent):
    """A cell agent with a position and a dispersal rate."""
    def __init__(self, model, cell, growth_rate):
        super().__init__(model) # make sure the parent class is initialized

        # assign properties to this instance explicitly
        self.cell = cell # this assigns the location of the agent at creation
        self.growth_rate = growth_rate # probability of the agent dispersing/hatching

    def hatch(self): # the behavior of the agent called  "hatch"
        if self.random.random() < self.growth_rate: # if some random number drawn is less (within) the assigned growth rate, then lets do stuff
            neighbors = list(self.cell.neighborhood.cells)  # identify the neighbors' cells of the current agent (self)
            empty_neighbors = [cell for cell in neighbors if len(cell.agents) == 0] # check which of those neighbors are empty
            if empty_neighbors: # if there are indeed empty neighbors (i.e., the list is not empty, and will thus evaluate to True)
                DispersalAgent(self.model, self.random.choice(empty_neighbors), self.growth_rate) # lets then make a new single agent. Simply calling the class name will do it. We pass in the model, a random choice of the empty neighbors, and the growth rate



## Model class
### Create the DispersalModel Class
Inherit from `mesa.Model` and implement:
* Constructor (__init__):
* Parameters: n, width, height, seed, growth_rate
* Inheret parent class constructor
* Create grid
* Filter starting positions to small rectangle (0,0) to (5,5)
* Create agents using `....create_agents()`
* Step method (step):

### Reminders
* `self.grid.all_cells.cells` - All grid cells
* `self.random.choices(list, k=n)` - Multiple random choices
* `self.agents.shuffle_do("method")` - Call method from a agents n random order

In [3]:
class DispersalModel(mesa.Model):
    """A model with number of agents, grid size, and random seed and dispersal rate."""
    def __init__(self, num_agents=4, width=25, height=25, growth_rate=0.2, seed=None):
        super().__init__(seed=seed)
        # assign propreties to this instance
        self.num_agents=num_agents
        self.width=width
        self.height=height
        self.growth_rate=growth_rate

        # make a space
        self.grid = OrthogonalMooreGrid((self.width, self.height), torus=False, random=self.random)

        # create agents
            # since we need to assign agents to cells, we first need to get a list of all possible cells that depends on our space
        possible_cells = self.grid.all_cells.cells
            # lets identify the first patch (6x6) of cells in the upper left corner (first we cheated a bit and just took the first 5 cells)
        cornerpatch_cells=[]
        for cell in possible_cells:
            if cell.coordinate[0] < 5 and cell.coordinate[1] < 5:
                cornerpatch_cells.append(cell)
            # create the agents             # how many?     # randomly select multipel (k) elements from a list        # agent needs a growth rate
        DispersalAgent.create_agents(self, self.num_agents, self.random.choices(cornerpatch_cells, k=self.num_agents), self.growth_rate)
        
    def step(self): # upon the model iterating a step, we are going to do things below
        self.agents.shuffle_do("hatch") # we call the behavior using the ,method that agents have, called shuffle_do, which has as input the behavior defined in the agent class (inputted as a string)
        

# Run visualization


In [4]:
# Agent visualization settings
space_viz = make_space_component() # make a space component with default settings (the space refers to the model's space)
# Setting Visualization settings (sliders etc.)
model = DispersalModel() # initialize a model

# visualize the model without any extra agent portrayal settings and only a space component
page = SolaraViz(model, [space_viz], name="Dispersal Agent")
page # call the visualization


Cannot show ipywidgets in text